# run_experiments：刺激 × 条件 × 解析 の一括実行

処理本体は `cisim.py`、比較・解析は `analysis.py` にあります。このノートブックは**設定してループを回し、結果を見るだけ**です。

| 足したいもの | やること |
|---|---|
| 入力音 | セル2の `STIMULI` に1行追加 |
| 処理条件（λ, ε など） | セル1の設定を変更 |
| 数値指標 | `analysis.py` に関数を書いて `METRICS` に登録 |
| 図 | セル5以降で `plot_xxx` を呼ぶ |

出力先は `results/<条件タグ>/` です（例: `results/seed0_eps0/`）。その下の `<キャリア>/<刺激>/lam<λ>/*.wav` に聴き比べ用の音声、`.../residual/` に差の信号、`metrics.csv`（`mode` 列でキャリアを区別）に全指標の表が保存されます。シード・ε を変えると別フォルダになるので上書きされません。出力ファイルの詳しい中身は `README.md` を参照してください。

**比較の考え方**：補償の残差は `vocoder`（拡散なし, G=I）を基準に見ます。`input` と直接比べると、ボコーダ自体の損失（TFSの欠落・雑音キャリア）が混ざってしまうためです。

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cisim as cs
import analysis as an
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")
plt.rcParams["font.family"] = "Hiragino Sans"   # Mac の標準日本語フォント
plt.rcParams["font.size"] = 10

## 1. 設定

In [ ]:
cfg = cs.CIConfig(sr=16000, seed=0)   # パラメータは cisim.CIConfig を参照

LAMBDAS = [2.0, 5.0, 9.0]   # 電流拡散 λ [mm]
EPS     = 0.0               # 逆行列の正則化 ε（0 = セクション5-2と同じ）
MODES   = ["noise", "tone"]  # キャリア: 帯域雑音（ノイズボコーダ）と正弦波（サイン波ボコーダ）を両方回す
MODE    = MODES[0]           # 1種類だけ使うセル（図・アブレーション）で使うキャリア
# 条件を変えたら結果を別フォルダに保存する（上書き防止）。例: results/seed0_eps0/
RUN_TAG = f"seed{cfg.seed}_eps{EPS:g}"
RESULTS = os.path.join("results", RUN_TAG)
print(cfg)

## 2. 刺激（入力音）

`名前: 関数` の形で登録します。ファイルは `data/` などに置いてパスを書いてください（使わないものはコメントアウトのままで大丈夫です）。

In [ ]:
sr = cfg.sr
STIMULI = {
    # --- テスト信号 ---
    "sweep":       lambda: cs.gen_sweep(sr, 100, 8000, dur=3.0),
    "sam_fm16":    lambda: cs.gen_sam(sr, fc=1000, fm=16),

    # --- 英語音声（同一文 EN0001） ---
    "en_m01":      lambda: cs.load_audio("data/speech_en/en_m01.wav", sr),
    "en_f01":      lambda: cs.load_audio("data/speech_en/en_f01.wav", sr),

    # --- 歌声（かたつむり, 最初の10秒） ---
    "song_m_low":  lambda: cs.load_audio("data/music_vocal/katatsumuri_m_low_jvs045.wav", sr, max_sec=10),
    "song_f_high": lambda: cs.load_audio("data/music_vocal/katatsumuri_f_high_jvs085.wav", sr, max_sec=10),

    # --- 楽器（歌声と同じ旋律, 最初の10秒） ---
    "piano_f_high": lambda: cs.gen_katatsumuri(sr, "f_high", "piano")[: sr * 10],
}
#STIMULI = {k: STIMULI[k] for k in ["sweep"]}   # 試し実行用。本番ではこの行を消す（または # を付ける）

# 比較する組（ref, test）。'input' は入力音そのもの
PAIRS = [
    ("input",   "vocoder"),     # ボコーダ自体の損失
    ("vocoder", "spread"),      # 電流拡散の影響
    ("vocoder", "comp_ideal"),  # 逆補償の残差（理想）
    ("vocoder", "comp_clip"),   # 逆補償の残差（電流≥0の制約あり）← 本題
    ("vocoder", "comp_nnls"),   # 逆補償の残差（NNLS）
    ("input",   "spread"),      # 元の音に対する劣化（補償なし）← キャリアの比較用
    ("input",   "comp_nnls"),   # 元の音に対する劣化（NNLS）    ← キャリアの比較用
]

## 3. 一括実行

In [ ]:
rows = []
OUTS = {}   # 図を描くために結果を保持 {(刺激, λ, キャリア): out}

for s_name, loader in STIMULI.items():
    x = loader()
    for lam in LAMBDAS:
        # 包絡・補償（NNLS を含む）の計算は1回だけ。キャリアは後で差し替える
        base = cs.run_conditions(x, cfg, lam, eps=EPS, mode=MODES[0])
        for mode in MODES:
            out = base if mode == MODES[0] else cs.resynthesize_with_carrier(base, cfg, mode)
            out["input"] = x
            OUTS[(s_name, lam, mode)] = out
            outdir = os.path.join(RESULTS, mode, s_name, f"lam{lam:g}")
            cs.save_wavs(outdir, x, out, cfg.sr)
            # 差の信号（test − vocoder）を residual/ に保存し、差の大きさ [dB] を受け取る
            res_db = cs.save_residuals(os.path.join(outdir, "residual"), out, cfg.sr)

            for ref_k, test_k in PAIRS:
                row = {"mode": mode, "stim": s_name, "lam": lam, "ref": ref_k, "test": test_k,
                       "neg_ratio": out["neg_ratio"]}
                row.update(an.compute_metrics(out[ref_k], out[test_k], cfg))
                # 包絡どうしの直接比較（vocoder基準のときだけ）
                if ref_k == "vocoder":
                    row.update(an.env_error(out["env"], out["env_" + test_k]))
                    row.update(an.env_contrast(out["env"], out["env_" + test_k]))
                    row["flatness"] = an.env_flatness(out["env_" + test_k])   # チャンネル方向の平坦度
                    row["flatness_ref"] = an.env_flatness(out["env"])          # 基準（拡散なし）の平坦度
                    row["residual_db"] = res_db.get(test_k, np.nan)            # 差の信号の大きさ
                rows.append(row)
    print("done:", s_name)

df_all = pd.DataFrame(rows)
os.makedirs(RESULTS, exist_ok=True)
df_all.to_csv(os.path.join(RESULTS, "metrics.csv"), index=False)
df = df_all[df_all["mode"] == MODE]   # これまでの表・図は MODE（ノイズ）の結果で描く
df_all.head()

## 4. 表で見る

In [ ]:
# 例: 補償後（comp_clip）の残差を 刺激 × λ で一覧
metric = "env_rel_err_db"   # 'snr_db', 'lsd_db', 'coh_mean', 'env_corr_mean' などに変更可
(df[df.test == "comp_clip"]
   .pivot_table(index="stim", columns="lam", values=metric)
   .round(2))

In [ ]:
# 例: 1つの刺激について全ペア × 全指標
df[(df.stim == "sweep") & (df.lam == 9.0)].set_index(["ref", "test"]).round(3)

## 5. 図で見る（刺激と λ を選んで描画）

In [ ]:
STIM, LAM = "sweep", 9.0
o = OUTS[(STIM, LAM, MODE)]

an.plot_spectrograms({k: o[k] for k in ["input", "vocoder", "spread", "comp_clip"]},
                     cfg.sr, diff_pair=("vocoder", "comp_clip"),
                     title=f"{STIM}  λ={LAM}mm");

In [ ]:
an.plot_ltas({k: o[k] for k in ["input", "vocoder", "spread", "comp_ideal", "comp_clip"]},
            cfg.sr, title=f"LTAS  {STIM}  λ={LAM}mm");

In [ ]:
an.plot_coherence(o["vocoder"], {k: o[k] for k in ["spread", "comp_ideal", "comp_clip"]},
                  cfg.sr, title=f"Coherence vs vocoder  {STIM}  λ={LAM}mm");

In [ ]:
# 包絡誤差ヒートマップ：補償誤差がどのch・どの時刻に出るか
an.plot_env_error(o["env"], o["env_comp_clip"], cfg, title=f"env error (comp_clip)  {STIM}  λ={LAM}mm");

In [ ]:
# 波形で見る（左=全体, 右=拡大）。全信号を同じ RMS にそろえ、縦軸も共通
# 最下段の「差」は residual/ の wav と同じ信号（comp_nnls − vocoder）
STIM_W, LAM_W, MODE_W = "en_m01", 9.0, MODE   # MODE_W = "tone" にすると正弦波キャリア
ZOOM = (1000, 1050)                           # 拡大する区間 [ms]（50 ms 程度だと1周期ずつ見える）
ow = OUTS[(STIM_W, LAM_W, MODE_W)]
w = {k: cs.rms_normalize(ow[k]) for k in ["input", "vocoder", "spread", "comp_nnls"]}
w["comp_nnls − vocoder"] = w["comp_nnls"] - w["vocoder"]
an.plot_waveforms(w, cfg.sr, zoom=ZOOM, sharey=True,
                  title=f"波形  {STIM_W}  λ={LAM_W}mm  キャリア={MODE_W}");

In [ ]:
os.makedirs("figures", exist_ok=True)
order = ["sweep", "sam_fm16", "en_m01", "en_f01", "song_m_low", "song_f_high", "piano_f_high"]
main  = ["sweep", "en_m01", "song_f_high", "piano_f_high"]

# 指標名: (縦軸ラベル, 表示する条件)
FIGS = {
    "contrast_ratio": ("コントラスト比\n(大きいほど良い)", ("spread", "comp_clip", "comp_nnls")),
    "profile_corr":   ("形の相関\n(大きいほど良い)",       ("spread", "comp_clip", "comp_nnls")),
    "env_rel_err_db": ("包絡誤差 [dB]\n(小さいほど良い)",  ("spread", "comp_nnls")),
}
for metric, (ylabel, tests) in FIGS.items():
    for name, stims in [("main", main), ("all", order)]:
        fig = an.plot_metric_vs_lambda(df, metric, stims, tests=tests, ylabel=ylabel)
        fig.savefig(f"figures/{metric}_{name}.png", dpi=200, bbox_inches="tight")
        plt.close(fig)   # 画面に大量に表示しないため

fig = an.plot_metric_vs_lambda(df, "stoi", ["en_m01", "en_f01"], ylabel="STOI\n(大きいほど良い)")
fig.savefig("figures/stoi.png", dpi=200, bbox_inches="tight")

## 5b. キャリアの比較（ノイズ vs 正弦波）

包絡と補償の計算は共通で、**音を作るときのキャリアだけ**が違います。そのため包絡の指標（包絡誤差・コントラスト比・形の相関・平坦度）は両者で同じ値になり、違いが出るのは**音声として評価する指標**（STOI など）と**聞こえ方**です。

聴き比べ：`results/<条件タグ>/noise/<刺激>/lam<λ>/` と `.../tone/<刺激>/lam<λ>/` の同じ番号のファイル。

In [ ]:
# 元の音声に対する STOI（英語音声）：キャリアごと・条件ごと
speech = ["en_m01", "en_f01"]
(df_all[(df_all.ref == "input") & df_all.stim.isin(speech)]
   .pivot_table(index=["stim", "test"], columns=["mode", "lam"], values="stoi")
   .round(3))

In [ ]:
# 補償の効果（拡散なしボコーダ音に対する STOI）がキャリアで変わるか
(df_all[(df_all.ref == "vocoder") & df_all.stim.isin(speech)]
   .pivot_table(index=["stim", "test"], columns=["mode", "lam"], values="stoi")
   .round(3))

In [ ]:
# スペクトログラムで見比べる（同じ刺激・λ・条件で、キャリアだけ違う）
STIM, LAM, COND = "en_m01", 9.0, "comp_nnls"
an.plot_spectrograms({"input": OUTS[(STIM, LAM, "noise")]["input"],
                      f"{COND} (noise)": OUTS[(STIM, LAM, "noise")][COND],
                      f"{COND} (tone)":  OUTS[(STIM, LAM, "tone")][COND]},
                     cfg.sr, title=f"キャリアの比較  {STIM}  λ={LAM}mm");

In [ ]:
# 波形でキャリアを比べる（同じ刺激・λ、同じ区間を拡大）
STIM_W, LAM_W, ZOOM = "en_m01", 9.0, (1000, 1050)
w = {"input": cs.rms_normalize(OUTS[(STIM_W, LAM_W, "noise")]["input"])}
for m in MODES:
    for k in ["vocoder", "comp_nnls"]:
        w[f"{k} ({m})"] = cs.rms_normalize(OUTS[(STIM_W, LAM_W, m)][k])
an.plot_waveforms(w, cfg.sr, zoom=ZOOM, sharey=True,
                  title=f"キャリアの比較（波形）  {STIM_W}  λ={LAM_W}mm");

## 6. アブレーション（残留ノイズの原因切り分け）の雛形

1つの要因だけを変えて残差がどう変わるかを見ます。例としてキャリアの乱数シードを振っています。
残差がシードで大きく変わるなら、原因はキャリア（雑音）由来です。他の要因（`n_ch`, `env_cut`, `MODE='tone'`, `EPS`）も同じ形で試せます。

In [ ]:
from dataclasses import replace

x = STIMULI["sweep"]()
abl = []
for seed in [0, 1, 2, 3, 4]:
    c = replace(cfg, seed=seed)
    o2 = cs.run_conditions(x, c, lam=9.0, eps=EPS, mode=MODE)
    r = an.compute_metrics(o2["vocoder"], o2["comp_clip"], c, which=["snr", "coherence"])
    r.update(an.env_error(o2["env"], o2["env_comp_clip"]))
    abl.append({"seed": seed, **r})
pd.DataFrame(abl).round(3)

## メモ：後で直す候補（`cisim.py` 内の `TODO(修正候補○)`）

| # | 内容 | 場所 |
|---|---|---|
| ① | Gを正規化してから逆行列を求める | `make_G_inv()` |
| ② | 事前補償の包絡が負になる（電流は負にできない） | `run_conditions` の `comp_clip` / `neg_ratio` で観察 |
| ③ | 4引数版 `resynthesize` はキャリアを毎回作り直していた | `run_conditions` でキャリアを共通化済み（元ノートとの違い） |
| ④ | ピーク正規化 → レベル比較ならRMS正規化 | `peak_normalize` |
| ⑤ | フィルタの ba形式・lfilter/filtfilt 混在 | `_band_coeffs`, `split_bands` |